# Feature extractors: basics

This tutorial introduces the `light_curve` feature extractor interface:
creating features, combining them with [`Extractor`](../api/meta/#light_curve.Extractor),
reading names and descriptions, and batch processing.

In [ ]:
# %pip install light-curve

## Setup

We create a synthetic sinusoidal light curve with Gaussian noise:

In [ ]:
import light_curve as licu
import numpy as np

rng = np.random.default_rng(42)
t = np.sort(rng.uniform(0, 100, 200))
m = 15.0 + 0.3 * np.sin(2 * np.pi * t / 20) + rng.normal(0, 0.05, 200)
err = np.full(200, 0.05)

print(f'Observations: {len(t)}')
print(f'Time range:   {t[0]:.1f} – {t[-1]:.1f} days')
print(f'Mag range:    {m.min():.2f} – {m.max():.2f}')

## Single feature

Each feature class is callable. It accepts `(t, m, sigma)` arrays and returns a NumPy array.
The `.names` attribute lists the output column names.

Here we use [`Amplitude`](../api/statistical/#light_curve.Amplitude) — the half peak-to-peak amplitude:

```sh
pip install light-curve
```

In [ ]:
import numpy as np

rng = np.random.default_rng(42)
t = np.sort(rng.uniform(0, 100, 200))
m = 15.0 + 0.3 * np.sin(2 * np.pi * t / 20) + rng.normal(0, 0.05, 200)
err = np.full(200, 0.05)

amp = licu.Amplitude()
result = amp(t, m, err)
print(f'names:  {amp.names}')
print(f'result: {result}')

`.descriptions` gives a human-readable explanation of each output:

In [ ]:
import light_curve as licu

amp = licu.Amplitude()
print(amp.descriptions)

## Combining features with `Extractor`

[`Extractor`](../api/meta/#light_curve.Extractor) combines multiple features into a single callable evaluated in one pass.
It is especially efficient for **cheap features** (statistical moments, variability
indices, etc.) because it avoids repeated passes over the data and reduces Python–Rust
call overhead:

In [ ]:
import light_curve as licu
import numpy as np

rng = np.random.default_rng(42)
t = np.sort(rng.uniform(0, 100, 200))
m = 15.0 + 0.3 * np.sin(2 * np.pi * t / 20) + rng.normal(0, 0.05, 200)
err = np.full(200, 0.05)

ext = licu.Extractor(
    licu.Amplitude(),
    licu.BeyondNStd(nstd=1),
    licu.BeyondNStd(nstd=2),
    licu.StandardDeviation(),
    licu.WeightedMean(),
    licu.LinearFit(),
    licu.StetsonK(),
)
result = ext(t, m, err)
for name, value in zip(ext.names, result):
    print(f'  {name:35s} = {value:.5f}')

## Batch processing with `.many()`

`.many()` processes a list of `(t, m, sigma)` tuples and returns a 2-D NumPy array
(shape `(N, n_features)`). It supports **multi-threading** (enabled by default via the
`n_jobs` parameter) and is the preferred path for large datasets:

```sh
pip install light-curve
```

In [ ]:
import light_curve as licu
import numpy as np

rng = np.random.default_rng(0)
light_curves = [
    (np.sort(rng.random(50)), rng.random(50), rng.random(50) * 0.1)
    for _ in range(1000)
]

results = licu.Amplitude().many(light_curves)
print(f'Extracted from {len(light_curves)} light curves: shape = {results.shape}')
print(f'Mean amplitude = {results.mean():.4f} mag')

## Batch processing with columnar libraries

`.many()` accepts any Arrow-compatible array — **PyArrow**, **Polars Series**, or
**nested-pandas** — as well as plain lists of `(t, m, sigma)` tuples.
All inputs must be in **`List<Struct<t, m, sigma>>`** format (one list entry per light curve).

### nested-pandas

[nested-pandas](https://nested-pandas.readthedocs.io) extends pandas with nested column support,
useful for working with catalog data like ZTF or Rubin LSST.

```sh
pip install light-curve nested-pandas s3fs universal-pathlib
```

In [ ]:
import light_curve as lc
import nested_pandas as npd
import numpy as np
import pyarrow as pa
from upath import UPath

# Read a ZTF DR23 HATS partition with nested light curves
s3_path = UPath(
    "s3://ipac-irsa-ztf/contributed/dr23/lc/hats/ztf_dr23_lc-hats/dataset/Norder=6/Dir=30000/Npix=34623/part0.snappy.parquet",
    anon=True,
)
ndf = npd.read_parquet(
    s3_path,
    columns=["objectid", "lightcurve.hmjd", "lightcurve.mag", "lightcurve.magerr"],
)

# Select rows with more than 10 observations
ndf = ndf.loc[ndf["lightcurve"].list_lengths > 10]

# Convert time to float32 so subcolumns share the same dtype
ndf["lightcurve.t"] = np.asarray(ndf["lightcurve.hmjd"] - 58000, dtype=np.float32)

# Extract features directly from the Arrow-backed nested column
feature = lc.Extractor(lc.Amplitude(), lc.EtaE(), lc.InterPercentileRange(quantile=0.25))
result = feature.many(pa.array(ndf["lightcurve"]), n_jobs=-1, arrow_fields=["t", "mag", "magerr"])

# Assign features back to the dataframe
ndf = ndf.assign(**dict(zip(feature.names, result.T)))
print(ndf.head())

### PyArrow

[PyArrow](https://arrow.apache.org/docs/python/) is the reference Python implementation of Apache Arrow.

```sh
pip install light-curve pyarrow
```

In [ ]:
import light_curve as lc
import numpy as np
import pyarrow as pa

rng = np.random.default_rng(42)
n_lc = 10

struct_type = pa.struct([("t", pa.float64()), ("m", pa.float64()), ("sigma", pa.float64())])
lcs_arrow = pa.array(
    [
        [{"t": t, "m": m, "sigma": s} for t, m, s in zip(*[np.sort(rng.random(50)) for _ in range(3)])]
        for _ in range(n_lc)
    ],
    type=pa.list_(struct_type),
)

feature = lc.Extractor(lc.Kurtosis(), lc.Skew(), lc.ReducedChi2())
result = feature.many(lcs_arrow, sorted=True, check=False, n_jobs=4, arrow_fields=["t", "m", "sigma"])
print(f"Features: {feature.names}")
print(f"Results shape: {result.shape}")

### Polars

[Polars](https://docs.pola.rs) is a fast DataFrame library built on Arrow.

```sh
pip install light-curve polars
```

In [ ]:
import light_curve as lc
import numpy as np
import polars as pl

rng = np.random.default_rng(42)

object_id = np.repeat(np.arange(10), 50)
t = np.sort(rng.random(500))
m = rng.random(500)
sigma = rng.random(500)

df = pl.DataFrame({"object_id": object_id, "t": t, "m": m, "sigma": sigma})

nested = df.group_by("object_id").agg(pl.struct("t", "m", "sigma").alias("lc"))

feature = lc.Extractor(lc.Amplitude(), lc.BeyondNStd(nstd=2), lc.LinearFit(), lc.StetsonK())
result = feature.many(nested["lc"], sorted=True, check=False, n_jobs=1, arrow_fields=["t", "m", "sigma"])

nested = nested.with_columns(
    [pl.Series(name, result[:, i]) for i, name in enumerate(feature.names)]
)
print(nested)

## Next steps

- [Feature table](../) — all 40+ extractors grouped by category
- [API reference](../api/) — full signatures and equations
- [Periodogram tutorial](../periodogram/) — Lomb–Scargle and period search